In [1]:
import pandas as pd
import requests
import logging
import os
from datetime import datetime
import json
import glob
import numpy as np
import sqlite3


**Extract**

In [2]:
# Making the request
url = "https://brasilapi.com.br/api/pix/v1/participants"

resposta = requests.get(url)

# Ckeking the response status
logging.info(f"Status Code: {resposta.status_code}")
logging.info(f"Iniciando extração da API {url}")

In [3]:
# Save the raw file to the raw folder
# This ensures traceability and allows for data reprocessing if business rules change
folder_raw = "../data/raw"
os.makedirs(folder_raw, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

caminho_arquivo = os.path.join(folder_raw, f"pix_raw_{timestamp}.json")

arquivo_json = resposta.json()

with open(caminho_arquivo, 'w', encoding='utf-8') as arquivo:
    # dump() take the variable and put it into the file
    # indent=4 this helps the file look nice and readable if you open it in Notepad
    json.dump(arquivo_json, arquivo, indent=4, ensure_ascii=False)

**Transform**

In [4]:
# Get the most recent JSON file
arquivos_raw = glob.glob(os.path.join(folder_raw, "*.json"))

if arquivos_raw:
    arquivos_raw.sort()
    arquivo_recente = arquivos_raw[-1]

    df = pd.read_json(arquivo_recente)
    linhas = len(df)
    print(f"Lendo o arquivo: {arquivo_recente}")
    print(f"Dados extraídos com sucesso: {linhas} registros encontrados.")
    logging.info(f"Dados extraídos com sucesso: {linhas} registros encontrados.")
else:
    print("Nenhum arquivo encontrado")

Lendo o arquivo: ../data/raw\pix_raw_20260622_174428.json
Dados extraídos com sucesso: 922 registros encontrados.


In [5]:
df.head()

,ispb,nome,nome_reduzido,modalidade_participacao,tipo_participacao,inicio_operacao
0,24313102,99PAY IP S.A.,99PAY IP S.A.,Provedor de Conta Transacional,Direta,NaN
1,35534511,A27 IP S/A,A27 IP S/A,Provedor de Conta Transacional,Indireta,NaN
2,48756121,A55 SCD S.A.,A55 SCD S.A.,Provedor de Conta Transacional,Direta,NaN
3,,ACCESSTAGE IP LTDA.,ACCESSTAGE IP LTDA.,Iniciador,,NaN
4,37715993,ACCREDITO SCD S.A.,ACCREDITO SCD S.A.,Provedor de Conta Transacional,Direta,NaN


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 922 entries, 0 to 921
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ispb                     921 non-null    object 
 1   nome                     921 non-null    object 
 2   nome_reduzido            921 non-null    object 
 3   modalidade_participacao  921 non-null    object 
 4   tipo_participacao        921 non-null    object 
 5   inicio_operacao          0 non-null      float64
dtypes: float64(1), object(5)
memory usage: 43.3+ KB


Limpeza

In [7]:
# Remove metadata/garbage rows from the API
df = df[df["modalidade_participacao"] != "Modalidade de Participação no Pix"]
df = df[df["tipo_participacao"] != "Tipo de Participação no SPI"]

In [8]:
# Replace the empty space with "Not informed"
df["tipo_participacao"] = df["tipo_participacao"].replace("",np.nan)
df["tipo_participacao"] = df["tipo_participacao"].fillna("Não informado")

# Normalize column names and institution text (no spaces at the ends and no double spaces in the middle)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip().str.replace(r'\s+', ' ', regex=True)

df

,ispb,nome,nome_reduzido,modalidade_participacao,tipo_participacao,inicio_operacao
0,24313102,99PAY IP S.A.,99PAY IP S.A.,Provedor de Conta Transacional,Direta,NaN
1,35534511,A27 IP S/A,A27 IP S/A,Provedor de Conta Transacional,Indireta,NaN
2,48756121,A55 SCD S.A.,A55 SCD S.A.,Provedor de Conta Transacional,Direta,NaN
3,,ACCESSTAGE IP LTDA.,ACCESSTAGE IP LTDA.,Iniciador,Não informado,NaN
4,37715993,ACCREDITO SCD S.A.,ACCREDITO SCD S.A.,Provedor de Conta Transacional,Direta,NaN
...,...,...,...,...,...,...
917,04088208,SEM PARAR IP,SEM PARAR IP,Provedor de Conta Transacional,Direta,NaN
918,20155248,UNIDA DTVM LTDA,UNIDA DTVM LTDA,Provedor de Conta Transacional,Direta,NaN
919,,VISA CONECTA IP LTDA.,VISA CONECTA IP LTDA.,Iniciador,Não informado,NaN
920,56106523,VUE IP S.A.,VUE IP S.A.,Provedor de Conta Transacional,Direta,NaN


Enriquecimento dos dados

In [9]:
# Creates a new institution category column based on the institution's name
MAPA_CATEGORIAS = {
    "Cooperativa": ["COOPERATIVA", "COOP", "CREDIT"],
    "Fintech / IP": ["IP S.A.", "INSTITUICAO DE PAGAMENTO", "PAGAMENTOS"],
    "Banco": ["BANCO", "BCO"]
    }

def classificar_instituicao(nome):
    # If the name is empty (NaN), play in Others
    if pd.isna(nome):
        return "Outros"

    nome_maiusculo = str(nome).upper()

    # Scan the dictionary looking for a match
    for categoria, termos in MAPA_CATEGORIAS.items():
        for termo in termos:
            if termo in nome_maiusculo:
                return categoria
    
    return "Outros"

df["categoria_instituicao"] = df["nome"].apply(classificar_instituicao)

df

,ispb,nome,nome_reduzido,modalidade_participacao,tipo_participacao,inicio_operacao,categoria_instituicao
0,24313102,99PAY IP S.A.,99PAY IP S.A.,Provedor de Conta Transacional,Direta,NaN,Fintech / IP
1,35534511,A27 IP S/A,A27 IP S/A,Provedor de Conta Transacional,Indireta,NaN,Outros
2,48756121,A55 SCD S.A.,A55 SCD S.A.,Provedor de Conta Transacional,Direta,NaN,Outros
3,,ACCESSTAGE IP LTDA.,ACCESSTAGE IP LTDA.,Iniciador,Não informado,NaN,Outros
4,37715993,ACCREDITO SCD S.A.,ACCREDITO SCD S.A.,Provedor de Conta Transacional,Direta,NaN,Cooperativa
...,...,...,...,...,...,...,...
917,04088208,SEM PARAR IP,SEM PARAR IP,Provedor de Conta Transacional,Direta,NaN,Outros
918,20155248,UNIDA DTVM LTDA,UNIDA DTVM LTDA,Provedor de Conta Transacional,Direta,NaN,Outros
919,,VISA CONECTA IP LTDA.,VISA CONECTA IP LTDA.,Iniciador,Não informado,NaN,Outros
920,56106523,VUE IP S.A.,VUE IP S.A.,Provedor de Conta Transacional,Direta,NaN,Fintech / IP


In [10]:
# Required columns
df = df[['ispb','nome','modalidade_participacao','tipo_participacao','categoria_instituicao']]
df

,ispb,nome,modalidade_participacao,tipo_participacao,categoria_instituicao
0,24313102,99PAY IP S.A.,Provedor de Conta Transacional,Direta,Fintech / IP
1,35534511,A27 IP S/A,Provedor de Conta Transacional,Indireta,Outros
2,48756121,A55 SCD S.A.,Provedor de Conta Transacional,Direta,Outros
3,,ACCESSTAGE IP LTDA.,Iniciador,Não informado,Outros
4,37715993,ACCREDITO SCD S.A.,Provedor de Conta Transacional,Direta,Cooperativa
...,...,...,...,...,...
917,04088208,SEM PARAR IP,Provedor de Conta Transacional,Direta,Outros
918,20155248,UNIDA DTVM LTDA,Provedor de Conta Transacional,Direta,Outros
919,,VISA CONECTA IP LTDA.,Iniciador,Não informado,Outros
920,56106523,VUE IP S.A.,Provedor de Conta Transacional,Direta,Fintech / IP


**Load**

In [11]:
# Save the dataframe to the SQLite database.
folder_processed = "../data/processed"
os.makedirs(folder_processed, exist_ok=True)

caminho_banco = os.path.join(folder_processed, "pix.db")

conexao = sqlite3.connect(caminho_banco)

df.to_sql('participantes_pix', conexao, if_exists='replace', index=False)

conexao.close()

In [12]:
total_bancos = df['nome'].nunique()
total_mod_part = df['modalidade_participacao'].nunique()
total_tipo_part = df['tipo_participacao'].nunique()

print(f'Total de bancos distintos: {total_bancos}')
print(f'Total de modalidades de participação distintas: {total_mod_part}')
print(f'Total de tipos de participação distintas: {total_tipo_part}')

Total de bancos distintos: 918
Total de modalidades de participação distintas: 4
Total de tipos de participação distintas: 3


In [13]:
df['modalidade_participacao'].unique()
df['tipo_participacao'].unique()

array(['Direta', 'Indireta', 'Não informado'], dtype=object)

In [14]:
cont_mod_part = df['modalidade_participacao'].value_counts()
cont_tipo_part = df['tipo_participacao'].value_counts()

print(cont_mod_part)

print(cont_tipo_part)

modalidade_participacao
Provedor de Conta Transacional    897
Iniciador                          19
Liquidante Especial                 3
Ente Governamental                  1
Name: count, dtype: int64
tipo_participacao
Indireta         674
Direta           224
Não informado     23
Name: count, dtype: int64
